# Sonde de prompt — le modèle peut-il choisir un outil jamais vu ?

**Runtime : GPU L4.** Secret : `HF_TOKEN`.

La sonde du 28/08 (`docs/tool_generalization_probe.md`) : le modèle lit la liste d'outils
(0 hallucination) mais **ne choisit jamais `delegate`** face à `web_search` — 0/4. Cause
plausible : la phrase d'instruction entraînée nomme ses deux outils et impose « au plus un ».

Ce notebook ne change **que cette phrase**, tout le reste fixe (mêmes énoncés, mêmes
listes d'outils) :
- `trained` : la phrase de l'entraînement (témoin) ;
- `generic` : « choisis l'outil dont la description correspond » ;
- `generic_delegate` : idem + précédence explicite de `delegate` sur les demandes multi-étapes.

Verdict attendu entre `===RESULT===` : routage `unseen` par prompt. Si `delegate` monte
à ≥ 3/4 avec un prompt, la couche agentique s'obtient **sans réentraîner** ; sinon → v6.


In [ ]:
# Jetons — le plus propre : Colab > icône clé > secrets HF_TOKEN (écriture), GEMINI_API_KEY, WANDB_API_KEY (optionnel).
import os
from getpass import getpass
try:
    from google.colab import userdata
    read = userdata.get
except Exception:
    read = lambda name: getpass(f"{name} : ")
for name, required in (("HF_TOKEN", True), ("GEMINI_API_KEY", False), ("WANDB_API_KEY", False)):
    try:
        value = read(name)
    except Exception:
        value = "" if not required else getpass(f"{name} : ")
    if value:
        os.environ[name] = value
    elif required:
        raise SystemExit(f"{name} manquant")
print("jetons chargés :", [n for n in ("HF_TOKEN", "GEMINI_API_KEY", "WANDB_API_KEY") if os.environ.get(n)])
g = os.environ.get("GEMINI_API_KEY", "")
if g:
    # Comparez avec la clé qui marche sur le poste : même préfixe, même longueur, sinon c'est un autre jeton.
    print(f"GEMINI_API_KEY : préfixe {g[:3]!r}, {len(g)} caractères")


## Lancer (≈ 25 min pour deux adaptateurs)

In [ ]:
# Mettez --adapters v4 si v5_1 n'est pas encore entraîné.
import os, subprocess, urllib.request
os.environ.update({
    "LFM2_BRANCH": "rd/pr_rca_eval_baseline",
    "LFM2_JOB": "tool_probe_prompts",
    "LFM2_ARGS": "--adapters v4,v5_1",
    "LFM2_EXTRAS": "serving-liquid,eval,inspect",
    "LFM2_ROOT": "/content/repo",
    "LFM2_OUT": "/content/out"
})
os.makedirs("/content/out", exist_ok=True)
urllib.request.urlretrieve("https://raw.githubusercontent.com/rcarvalo/finetuning_s2s_toolcalling/rd/pr_rca_eval_baseline/infra/colab_entrypoint.sh", "/content/entry.sh")

# Au PREMIER PLAN, volontairement : le kernel occupé est ce qui garde la session Colab vivante.
# La sortie est streamée ligne à ligne ici ET dans /content/out/job.log ; les lignes ===RESULT===
# sont reprises en résumé à la fin, avec le VRAI code de sortie du job.
results = []
with subprocess.Popen(["bash", "/content/entry.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
        if "===RESULT===" in line:
            results.append(line.strip())
code = proc.returncode
print("\n" + "=" * 70)
print("STATUT :", "SUCCÈS" if code == 0 else f"ÉCHEC (code {code}) — voir les dernières lignes ci-dessus")
for line in results:
    print("  ", line)
print("journal complet : /content/out/job.log")
